# Fiction Forks: WORLDLINE検証ノートブック

公開GitHub branchをcloneし、unit test、2036年比較、5役×3ターンfixtureを実行します。GitHub token、API key、live AI providerは使いません。PR作成や外部書込みも行いません。

In [ ]:
repository = "https://github.com/nexus-ai-2045/fiction-forks.git" # @param {type:"string"}
branch = "main" # @param {type:"string"}
slug = "doraemon-public-tools" # @param {type:"string"}

import re

if repository != "https://github.com/nexus-ai-2045/fiction-forks.git":
    raise ValueError("このnotebookはFiction Forksの公開repositoryだけを対象にします。")
if not re.fullmatch(r"[A-Za-z0-9._/-]{1,120}", branch) or ".." in branch:
    raise ValueError("branch名が安全な形式ではありません。")
if not re.fullmatch(r"[a-z0-9](?:[a-z0-9-]{0,62}[a-z0-9])?", slug):
    raise ValueError("slugは小文字英数字とhyphenだけで指定してください。")


In [ ]:
from pathlib import Path
import shutil
import subprocess

workspace = Path("/content/fiction-forks")
if workspace.exists():
    shutil.rmtree(workspace)
subprocess.run(["git", "clone", "--depth", "1", "--branch", branch, repository, str(workspace)], check=True)
print(f"checked out: {branch}")


In [ ]:
import os

environment = os.environ.copy()
environment["PYTHONUTF8"] = "1"
environment["PYTHONPATH"] = "src"
subprocess.run(["python", "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=workspace, env=environment, check=True)


In [ ]:
scenario = "scenarios/japan-2036/scenario.json"
intervention = f"interventions/{slug}.json"
social_config = f"scenarios/japan-2036/social-{slug}.json"
fixture = f"fixtures/social/{slug}.jsonl"

# 0.2.0の最初の世界線だけは旧pathを保ちます。新規PRはslug付きpathを使います。
if slug == "doraemon-public-tools" and not (workspace / social_config).exists():
    social_config = "scenarios/japan-2036/social.json"
    fixture = "fixtures/social/japan-2036-cooperation.jsonl"

for relative in (scenario, intervention, social_config, fixture):
    if not (workspace / relative).is_file():
        raise FileNotFoundError(relative)

comparison = subprocess.run(
    ["python", "-m", "fiction_forks", "compare", "--scenario", scenario, "--intervention", intervention, "--seed", "2036"],
    cwd=workspace, env=environment, check=True, capture_output=True, text=True, encoding="utf-8"
)
print(comparison.stdout)


In [ ]:
artifact = workspace / "colab-worldline-run.json"
subprocess.run(
    ["python", "-m", "fiction_forks", "social", "--scenario", scenario, "--intervention", intervention, "--social-config", social_config, "--provider", "fixture", "--fixture", fixture, "--output", str(artifact)],
    cwd=workspace, env=environment, check=True
)
print(f"artifact: {artifact}")
print("fixtureはプロトコル検証であり、live LLM実測ではありません。")
